# Evaluación de Modelos y Selección de la Métrica Campeona

1. **¿Cómo sabemos si un modelo es bueno?**
2. **¿Por qué Accuracy (Exactitud) es una trampa peligrosa?**
3. **¿Qué es el F1-Score y la Matriz de Confusión?**
4. **¿Cómo interpretamos las Curvas ROC y el valor AUC?**

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import tensorflow as tf

# Importar herramientas de evaluación
from sklearn.metrics import auc, classification_report, confusion_matrix, roc_curve

# Añadir directorio raíz al path
sys.path.append(str(Path.cwd().parent))

# Configuración visual
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

## ¿Accuracy o F1-Score?

* **Accuracy (Exactitud)**: Mide cuántas predicciones en total fueron correctas. Si el 80% de tus clientes no cancela, un modelo que diga "nadie cancelará" tendría un 80% de exactitud. Fantastico pero inutil.

* **F1-Score**: Es el promedio entre **Precision** (de las reservas que se predice que se cancelaran, cuántas se llegan a cancelan realmente) y **Recall** (de todas las que realmente se cancelaron, cuántas logramos "atrapar"). F1-Score es perfecto cuando hay un desbalanceo moderado a severo.

## 1. Cargar Datos de Prueba (Test)

Cargamos los datos y aplicamos la división exactamente como en el pipeline de entrenamiento para asegurar la eleccion.

In [ ]:
from sklearn.model_selection import train_test_split

from src.config import COLS_TO_DROP, MODELS_DIR, OUTPUTS_DIR, RAW_DATA_PATH, TARGET_COL

df = pd.read_csv(RAW_DATA_PATH)
df_clean = df.drop(columns=[col for col in COLS_TO_DROP if col in df.columns])
X = df_clean.drop(columns=[TARGET_COL])
y = df_clean[TARGET_COL]

_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Conjunto de evaluación cargado. Tamaño del Test: {X_test.shape[0]:,} registros.")

## 2. Evaluar Todos los Modelos Entrenados

Función robusta para cargar cada modelo desde la carpeta `/models`, realizar predicciones probabilísticas y binarias, y calcular las métricas.

In [ ]:
def evaluate_saved_model(model_name, X_val, y_val):
    """Carga un modelo entrenado y retorna sus métricas y predicciones."""
    print(f"Cargando y evaluando {model_name}...")

    if model_name == "neural_network":
        # Componentes de Deep Learning
        preprocessor = joblib.load(MODELS_DIR / "nn_preprocessor.pkl")
        keras_clf = tf.keras.models.load_model(MODELS_DIR / "neural_network.keras")

        # Transformar datos manualmente antes de pasar a Keras
        X_processed = preprocessor.transform(X_val)
        y_probs = keras_clf.predict(X_processed).flatten()
        y_preds = (y_probs >= 0.5).astype(int)
    else:
        # Pipeline clásico completo (incluye preprocesador interno)
        pipeline = joblib.load(MODELS_DIR / f"{model_name}.pkl")
        y_preds = pipeline.predict(X_val)

        if hasattr(pipeline, "predict_proba"):
            y_probs = pipeline.predict_proba(X_val)[:, 1]
        else:
            y_probs = y_preds  # Fallback

    # Generar métricas detalladas
    report = classification_report(y_val, y_preds, output_dict=True)
    accuracy = report["accuracy"]
    f1 = report["weighted avg"]["f1-score"]

    # Calcular Curva ROC
    fpr, tpr, _ = roc_curve(y_val, y_probs)
    roc_auc = auc(fpr, tpr)

    return {
        "accuracy": accuracy,
        "f1_score": f1,
        "roc_auc": roc_auc,
        "fpr": fpr,
        "tpr": tpr,
        "y_preds": y_preds,
        "y_probs": y_probs,
    }

In [ ]:
models_to_evaluate = [
    "logistic_regression",
    "decision_tree",
    "random_forest",
    "xgboost",
    "neural_network",
]

evaluation_results = {}

for name in models_to_evaluate:
    evaluation_results[name] = evaluate_saved_model(name, X_test, y_test)

## 3. Construir la Tabla Comparativa de Resultados

Creamos un DataFrame de Pandas para contrastar las métricas clave de cada modelo.

In [ ]:
records = []
for name, metrics in evaluation_results.items():
    records.append(
        {
            "Modelo": name.replace("_", " ").title(),
            "Accuracy (Exactitud)": f"{metrics['accuracy']:.4%}",
            "F1-Score (Equilibrio)": f"{metrics['f1_score']:.4%}",
            "ROC-AUC (Área Curva)": f"{metrics['roc_auc']:.4f}",
        }
    )

df_comparison = pd.DataFrame(records)
# Guardamos la comparativa en CSV en outputs/
df_comparison.to_csv(OUTPUTS_DIR / "comparativa_modelos.csv", index=False)
print("\n--- REPORT COMPARATIVO FINAL ---")
df_comparison

> **XGBoost** o **Random Forest** lideran la tabla con un F1-Score superior. Son algoritmos basados en ensamblados de árboles de decisión (*Ensemble Learning*), los campeones indiscutibles en datos tabulares debido a su capacidad para manejar no-linealidades sin requerir preprocesamientos extremos.

## 4. Visualización de Resultados Requeridos

### A. Matriz de Confusión (Eg: Random Forest / XGBoost)

La matriz de confusión nos detalla:
* **True Negatives (TN)**: Clientes que predijimos que harían Check-in y lo hicieron.
* **False Positives (FP)**: Clientes que predijimos que cancelarían pero hicieron Check-in (falsa alarma).
* **False Negatives (FN)**: Clientes que predijimos que harían Check-in pero cancelaron (**el error más costoso para el hotel**).
* **True Positives (TP)**: Reservas canceladas que detectamos con éxito.

In [ ]:
# Seleccionamos el modelo ganador (eg: xgboost)
target_model = "xgboost"
y_preds = evaluation_results[target_model]["y_preds"]

cm = confusion_matrix(y_test, y_preds)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["No Cancela (0)", "Cancela (1)"],
    yticklabels=["No Cancela (0)", "Cancela (1)"],
    annot_kws={"fontsize": 14, "weight": "bold"},
)
plt.title(f"Matriz de Confusión de {target_model.upper()}", fontsize=15, pad=15)
plt.ylabel("Realidad", fontsize=13)
plt.xlabel("Predicción del Modelo", fontsize=13)
plt.tight_layout()
plt.show()

### B. Curvas ROC Comparativas

La Curva ROC (Receiver Operating Characteristic) grafica la Sensibilidad (Verdaderos Positivos) frente a 1 - Especificidad (Falsos Positivos) para todos los umbrales de decisión posibles.

* Un modelo perfecto tiene un **AUC = 1.0** (línea que sube directo a 1 y luego va recta a la derecha).
* Un modelo aleatorio (una moneda al aire) sigue la diagonal punteada negra con un **AUC = 0.5**.

In [ ]:
plt.figure(figsize=(10, 8))

for name, res in evaluation_results.items():
    label_name = name.replace("_", " ").title()
    plt.plot(
        res["fpr"], res["tpr"], label=f"{label_name} (AUC = {res['roc_auc']:.4f})", linewidth=2.5
    )

# Dibujar la línea de decisión aleatoria
plt.plot([0, 1], [0, 1], "k--", label="Decisión al Azar (AUC = 0.5000)", alpha=0.7)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("Tasa de Falsos Positivos (1 - Especificidad)", fontsize=12)
plt.ylabel("Tasa de Verdaderos Positivos (Sensibilidad)", fontsize=12)
plt.title("Comparativa de Curvas ROC (Receiver Operating Characteristic)", fontsize=15, pad=15)
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

# Guardar el gráfico comparativo final en la carpeta de outputs
plt.savefig(OUTPUTS_DIR / "roc_curve_comparison.png")
plt.show()